# V10ADgenevalidation — independent external validation of the gene lists

**Question.** Do the genes that PLS-DA and the CVAE selected hold up in brains the
pipeline has never seen, beyond what any list of AD-associated genes would do?

**Frozen lists.** The gene lists are read from the finished **V9ADexternal** run,
where they were selected on four training cohorts only (GSE48350, GSE33000,
GSE118553, GSE84422). A SHA-256 fingerprint of the list file is printed before any
validation cohort is loaded. Nothing below can change which genes are tested.

| set | genes | defined on | tested on |
|---|---|---|---|
| PLS-DA | all genes PLS-DA selected | 4 training cohorts | every validation cohort |
| CVAE | all genes the CVAE selected | 4 training cohorts | every validation cohort |
| Core | genes selected by **both** models **and** replicated in GSE132903 | training + GSE132903 | the new cohorts only (never GSE132903) |

**Validation cohorts** (none used for training or selection; different brain banks,
countries and platforms):

| cohort | test | donors | region | platform |
|---|---|---|---|---|
| GSE132903 | AD vs control | 195 | middle temporal gyrus | Illumina HT-12 v4 |
| GSE15222 | AD vs control | 363 | temporal cortex | Illumina RefSeq-8 |
| GSE29378 | AD vs control | 33 | hippocampus CA1 + CA3 | Illumina HT-12 v3 |
| GSE122063 | AD vs control | 23 | frontal + temporal cortex | Agilent |
| GSE36980 | AD vs control | 33 | frontal cortex | Affymetrix HuGene 1.0 ST |
| GSE106241 | dose-response: Braak 0-6 | 60 | inferior temporal cortex | Agilent |
| GSE1297 | dose-response: MMSE, tangles, Braak | 31 | hippocampus | Affymetrix U133A |

GSE5281 is not used: it profiles laser-captured neurons only, so glial genes such as
*GFAP* cannot replicate there by construction.

**Method, fixed in advance.**
1. Every cohort gets the project's preprocessing (probe → gene via NCBI Gene,
   unmapped / multi-gene probes dropped, probes of one gene averaged), AD vs control
   only, donors aged ≥ 60.
2. **One donor = one observation**: arrays from several regions or technical replicates
   of a donor are averaged.
3. Per gene, a linear model `expression ~ diagnosis + age + sex` gives an **age- and
   sex-adjusted** standardised effect and its standard error; BH-FDR within cohort.
4. **Random-effects meta-analysis** (DerSimonian-Laird) across the AD-vs-control cohorts:
   pooled effect, 95% CI, I².
5. Each set is scored three ways — direction concordance, replication (same direction
   and FDR < 0.05), and a model-free signature score (mean of up genes minus mean of down
   genes) — and compared with **two baselines**:
   * **matched random**: 2,000 sets of genes matched gene-for-gene on training effect
     size (decile bins), i.e. equally AD-associated in training;
   * **plain DE**: the top-N genes by training differential expression, the simplest
     competitor a reviewer would ask about.
6. **Dose-response**: signature score against Braak stage, MMSE and tangle density,
   adjusted for age (partial Spearman), with the matched-random null.


In [ ]:
# ======================================================================
# Downloads and parsers
# ======================================================================
import os, re, io, gzip, glob, requests, numpy as np, pandas as pd
import matplotlib.pyplot as plt

OUT, TMP = "/kaggle/working", "/kaggle/tmp"
os.makedirs(TMP, exist_ok=True)
AGE_MIN = 60
HDR = {"User-Agent": "ad-merged-geo-cohorts (kaggle)"}


def fetch(url, name):
    """Stream a file to Kaggle's scratch disk, with retries."""
    path = f"{TMP}/{name}"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    for attempt in range(5):
        try:
            with requests.get(url, headers=HDR, stream=True, timeout=300) as r:
                r.raise_for_status()
                with open(path, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            print(f"downloaded {name}: {os.path.getsize(path)/1e6:.1f} MB")
            return path
        except Exception as e:                       # noqa: BLE001
            print(f"  retry {attempt+1} for {name}: {e}")
    raise RuntimeError(f"could not download {url}")


def series_matrix(acc, suffix=""):
    stub = acc[:-3] + "nnn"
    fn = f"{acc}{suffix}_series_matrix.txt.gz"
    return fetch(f"https://ftp.ncbi.nlm.nih.gov/geo/series/{stub}/{acc}/matrix/{fn}", fn)


def read_series_matrix(path):
    """-> (expression DataFrame probes x GSM, per-sample metadata DataFrame)."""
    meta_lines, table = [], []
    with gzip.open(path, "rt", errors="ignore") as f:
        in_table = False
        for line in f:
            if line.startswith("!series_matrix_table_begin"):
                in_table = True; continue
            if line.startswith("!series_matrix_table_end"):
                break
            (table if in_table else meta_lines).append(line)
    expr = pd.read_csv(io.StringIO("".join(table)), sep="\t", index_col=0)
    expr.index = expr.index.astype(str)
    expr = expr.apply(pd.to_numeric, errors="coerce")

    gsm = None; rows = {}
    for line in meta_lines:
        parts = [p.strip().strip('"') for p in line.rstrip("\n").split("\t")]
        key, vals = parts[0], parts[1:]
        if key == "!Sample_geo_accession":
            gsm = vals
        elif key in ("!Sample_title", "!Sample_source_name_ch1"):
            rows[key[8:]] = vals
        elif key.startswith("!Sample_characteristics_ch"):
            for i, v in enumerate(vals):
                if ":" in v:
                    k, val = v.split(":", 1)
                    rows.setdefault(k.strip().lower(), [None] * len(vals))[i] = val.strip()
    meta = pd.DataFrame(rows, index=gsm)
    meta.index.name = "gsm"
    return expr, meta


def platform_table(gpl):
    path = fetch(f"https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={gpl}&targ=self&form=text&view=data",
                 f"{gpl}.txt")
    lines, on = [], False
    with open(path, errors="ignore") as f:
        for line in f:
            if line.startswith("!platform_table_begin"):
                on = True; continue
            if line.startswith("!platform_table_end"):
                break
            if on:
                lines.append(line)
    t = pd.read_csv(io.StringIO("".join(lines)), sep="\t", dtype=str)
    t["ID"] = t["ID"].astype(str)
    return t.set_index("ID")


def maybe_log2(df, name):
    """GEO2R's rule for whether a matrix still needs log2."""
    qx = np.nanquantile(df.values.astype(float), [0, .25, .5, .75, .99, 1.0])
    need = (qx[4] > 100) or (qx[5] - qx[0] > 50 and qx[1] > 0)
    print(f"{name}: quantiles {np.round(qx, 2)} -> {'log2 applied' if need else 'already log scale'}")
    if need:
        df = np.log2(df.clip(lower=0) + 1)
    return df

# ---- NCBI Gene: Entrez ID -> current official symbol; symbol/synonym -> Entrez
gi = pd.read_csv(fetch("https://ftp.ncbi.nlm.nih.gov/gene/DATA/GENE_INFO/Mammalia/Homo_sapiens.gene_info.gz",
                       "Homo_sapiens.gene_info.gz"), sep="\t", dtype=str,
                 usecols=["GeneID", "Symbol", "Synonyms"])
ENTREZ2SYM = dict(zip(gi.GeneID, gi.Symbol))
OFFICIAL = set(gi.Symbol)
_syn = {}
for gid, syns in zip(gi.GeneID, gi.Synonyms):
    if isinstance(syns, str) and syns != "-":
        for s in syns.split("|"):
            _syn.setdefault(s, set()).add(gid)
SYN2SYM = {s: ENTREZ2SYM[next(iter(g))] for s, g in _syn.items() if len(g) == 1 and s not in OFFICIAL}
print(f"NCBI Gene: {len(OFFICIAL):,} official symbols, {len(SYN2SYM):,} unambiguous synonyms")


def norm_symbol(s):
    if not isinstance(s, str) or not s or s == "-":
        return None
    s = s.strip()
    if s in OFFICIAL:
        return s
    return SYN2SYM.get(s)


def probes_to_symbols(expr, ann, entrez_col=None, symbol_col=None, name=""):
    """Project recipe: probe -> one gene; unmapped / multi-gene probes dropped;
    probes of the same gene averaged."""
    sym = pd.Series(index=ann.index, dtype=object)
    if entrez_col is not None:
        ez = ann[entrez_col].fillna("")
        single = ~ez.str.contains("///") & ez.str.match(r"^\d+$")
        sym[single] = ez[single].map(ENTREZ2SYM)
    if symbol_col is not None:
        sy = ann[symbol_col].fillna("")
        fill = sym.isna() & ~sy.str.contains("///") & (sy != "")
        sym[fill] = sy[fill].map(norm_symbol)
    sym = sym.reindex(expr.index)
    keep = sym.notna()
    print(f"{name}: {len(expr):,} probes -> {keep.sum():,} single-gene probes -> "
          f"{sym[keep].nunique():,} genes")
    return expr[keep].groupby(sym[keep]).mean()

print("parsers ready")

def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float); n = len(pvals)
    order = np.argsort(pvals)
    q = pvals[order] * n / np.arange(1, n + 1)
    q = np.clip(np.minimum.accumulate(q[::-1])[::-1], 0, 1)
    out = np.empty(n); out[order] = q
    return out


In [ ]:
# ======================================================================
# Frozen gene lists from V9ADexternal, and training-cohort DE statistics
# ======================================================================
import hashlib, glob
from scipy import stats
from scipy.stats import spearmanr, rankdata
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

AGE_MIN = 60
N_NULL = 2000
RNG = np.random.RandomState(2026)
TRAIN_COHORTS = ["GSE48350", "GSE33000", "GSE118553", "GSE84422"]

_sel = glob.glob("/kaggle/input/**/loco_selected_genes.csv", recursive=True)
assert _sel, "attach the v9adexternal notebook output as an input"
SEL_PATH = _sel[0]
print("frozen gene lists:", SEL_PATH)
print("SHA-256:", hashlib.sha256(open(SEL_PATH, "rb").read()).hexdigest())
sel = pd.read_csv(SEL_PATH)
sel["replicated"] = sel.replicated.astype(str).str.lower().eq("true")
sel = sel[sel.direction_train.isin(["up", "down"])]
SIGN = {"up": 1, "down": -1}

SETS = {}
for m in ("PLS-DA", "CVAE"):
    s = sel[sel.method == m]
    SETS[m] = dict(zip(s.gene, s.direction_train.map(SIGN)))
core = (set(sel[(sel.method == "PLS-DA") & sel.replicated].gene)
        & set(sel[(sel.method == "CVAE") & sel.replicated].gene))
SETS["Core"] = {g: SETS["PLS-DA"][g] for g in sorted(core)}
for k, v in SETS.items():
    print(f"{k:7s} {len(v):4d} genes  ({sum(x > 0 for x in v.values())} up / {sum(x < 0 for x in v.values())} down)")
print("Core genes:", ", ".join(SETS["Core"]))

# training statistics, recomputed from the merged matrix on the four training cohorts
_m = glob.glob("/kaggle/input/**/merged_expression.csv.gz", recursive=True)
assert _m, "attach the ad-merged-geo-cohorts notebook output as an input"
MDIR = os.path.dirname(_m[0])
MX = pd.read_csv(f"{MDIR}/merged_expression.csv.gz", index_col=0)
MM = pd.read_csv(f"{MDIR}/merged_metadata.csv", index_col=0).loc[MX.index]
tr = MM.cohort.isin(TRAIN_COHORTS).values
Xt, yt = MX.values[tr], (MM.dx.values[tr] == "AD")
TRAIN_LFC = pd.Series(Xt[yt].mean(0) - Xt[~yt].mean(0), index=MX.columns)
TRAIN_P = pd.Series(stats.ttest_ind(Xt[yt], Xt[~yt], equal_var=False).pvalue, index=MX.columns)
print(f"training statistics: {tr.sum()} arrays, {MX.shape[1]:,} genes")


In [ ]:
# ======================================================================
# Validation cohorts: download, preprocess, collapse to one row per donor
# ======================================================================
def finish(e, m, name, platform):
    """e: genes x arrays (log scale), m: per-array metadata with donor, dx, age, sex.
    Keeps AD/control (or severity), age >= 60, averages arrays per donor."""
    m = m[m.index.isin(e.columns)].copy()
    m = m[m.age.isna() | (m.age >= AGE_MIN)]
    e = e[m.index]
    num = [c for c in m.columns if c not in ("donor", "dx", "sex", "region")]
    d_expr = e.T.groupby(m.donor.values).mean()
    d_expr = d_expr.loc[:, d_expr.notna().mean() >= 0.8]
    d_expr = d_expr.fillna(d_expr.mean())
    d_meta = m.groupby("donor").agg({**{c: "mean" for c in num},
                                     **{c: "first" for c in ("dx", "sex", "region") if c in m}})
    d_meta = d_meta.loc[d_expr.index]
    d_meta["cohort"] = name; d_meta["platform"] = platform
    d_meta["n_arrays"] = m.groupby("donor").size().loc[d_expr.index].values
    print(f"{name}: {len(m)} arrays -> {len(d_expr)} donors, {d_expr.shape[1]:,} genes; "
          f"dx {d_meta.dx.value_counts(dropna=False).to_dict()}")
    return d_expr, d_meta


def raw_characteristics(acc):
    """Characteristics lines verbatim (for series that do not use 'key: value')."""
    rows = []
    with gzip.open(series_matrix(acc), "rt", errors="ignore") as f:
        for line in f:
            if line.startswith("!Sample_characteristics_ch1"):
                rows.append([p.strip().strip('"') for p in line.rstrip("\n").split("\t")[1:]])
            if line.startswith("!series_matrix_table_begin"):
                break
    return [",".join(x) for x in zip(*rows)]


def sex_std(v):
    v = str(v).strip().lower()
    return "male" if v in ("m", "male", "2") else "female" if v in ("f", "female", "1") else np.nan


VAL = {}

def load_gse132903():
    """Already preprocessed in the merged matrix (one array per donor)."""
    sub = MM[MM.cohort == "GSE132903"]
    e = MX.loc[sub.index].T
    m = pd.DataFrame({"donor": sub.donor, "dx": sub.dx, "age": sub.age,
                      "sex": sub.sex.map(sex_std), "region": sub.region}, index=sub.index)
    return finish(e, m, "GSE132903", "GPL10558")


def load_gse15222():
    expr, meta = read_series_matrix(series_matrix("GSE15222"))
    ann = platform_table("GPL2700")
    # GPL2700 carries RefSeq accessions only: map them with UCSC's RefSeq tables
    acc2sym = {}
    for url, name in (("https://hgdownload.soe.ucsc.edu/goldenPath/hg19/database/refGene.txt.gz", "refGene_hg19.txt.gz"),
                      ("https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/ncbiRefSeq.txt.gz", "ncbiRefSeq_hg38.txt.gz")):
        t = pd.read_csv(fetch(url, name), sep="\t", header=None, usecols=[1, 12], dtype=str)
        for acc, sym in zip(t[1].str.split(".").str[0], t[12]):
            acc2sym.setdefault(acc, set()).add(sym)
    gb = ann["GB_ACC"].fillna("").str.split(".").str[0]
    ann = ann.assign(sym=[next(iter(acc2sym[a])) if len(acc2sym.get(a, ())) == 1 else None for a in gb])
    e = probes_to_symbols(maybe_log2(expr, "GSE15222"), ann, symbol_col="sym", name="GSE15222")
    raw = pd.Series(raw_characteristics("GSE15222"), index=meta.index)
    title = meta["title"] if "title" in meta else pd.Series(meta.index, index=meta.index)
    age = pd.to_numeric(raw.str.extract(r"AGE_(\d+)", flags=re.I)[0], errors="coerce")
    sex = raw.str.extract(r"GENDER_(\w+)", flags=re.I)[0].map(sex_std)
    dx = np.where(title.str.upper().str.startswith("WGAAD"), "AD",
                  np.where(title.str.upper().str.startswith("WGACON"), "Control", None))
    m = pd.DataFrame({"donor": meta.index, "dx": dx, "age": age, "sex": sex,
                      "region": "temporal cortex"}, index=meta.index)
    return finish(e, m.dropna(subset=["dx"]), "GSE15222", "GPL2700")


def load_gse29378():
    expr, meta = read_series_matrix(series_matrix("GSE29378"))
    e = probes_to_symbols(maybe_log2(expr, "GSE29378"), platform_table("GPL6947"),
                          entrez_col="Entrez_Gene_ID", symbol_col="Symbol", name="GSE29378")
    dx = meta["title"].str.split(" - ").str[0].str.strip().map({"AD": "AD", "Control": "Control"})
    m = pd.DataFrame({"donor": "S" + meta["subjectnumber"].astype(str), "dx": dx,
                      "age": pd.to_numeric(meta["age(years)"], errors="coerce"),
                      "sex": meta["gender"].map(sex_std),
                      "braak": pd.to_numeric(meta["braakstage(0-6)"], errors="coerce"),
                      "plaque": pd.to_numeric(meta["plaquescore(0-3)"], errors="coerce"),
                      "region": "hippocampus"}, index=meta.index)
    return finish(e, m.dropna(subset=["dx"]), "GSE29378", "GPL6947")


def load_gse122063():
    expr, meta = read_series_matrix(series_matrix("GSE122063"))
    e = probes_to_symbols(maybe_log2(expr, "GSE122063"), platform_table("GPL16699"),
                          entrez_col="LOCUSLINK_ID", symbol_col="GENE_SYMBOL", name="GSE122063")
    dx = meta["patient diagnosis"].map({"Alzheimer's disease": "AD", "Control": "Control"})   # VaD dropped
    m = pd.DataFrame({"donor": "S" + meta["subject id"].astype(str), "dx": dx,
                      "age": pd.to_numeric(meta["age"], errors="coerce"),
                      "sex": meta["sex"].map(sex_std), "region": "frontal + temporal"}, index=meta.index)
    return finish(e, m.dropna(subset=["dx"]), "GSE122063", "GPL16699")


def load_gse36980():
    expr, meta = read_series_matrix(series_matrix("GSE36980"))
    ann = platform_table("GPL6244")
    def one_entrez(s):
        if not isinstance(s, str) or s.strip() in ("", "---"):
            return None
        ids = {p.split("//")[-1].strip() for p in s.split("///") if "//" in p}
        ids = {i for i in ids if i.isdigit()}
        return ids.pop() if len(ids) == 1 else None
    ann = ann.assign(entrez=ann["gene_assignment"].map(one_entrez))
    e = probes_to_symbols(maybe_log2(expr, "GSE36980"), ann, entrez_col="entrez", name="GSE36980")
    t = meta["title"]
    fc = t.str.contains("_FC")                          # frontal cortex only: donors are not linked across regions
    dx = np.where(t.str.startswith("non-AD"), "Control", np.where(t.str.startswith("AD"), "AD", None))
    m = pd.DataFrame({"donor": meta.index, "dx": dx,
                      "age": pd.to_numeric(meta["age"], errors="coerce"),
                      "sex": meta["sex"].map(sex_std), "region": "frontal cortex"}, index=meta.index)
    return finish(e, m[fc.values].dropna(subset=["dx"]), "GSE36980", "GPL6244")


def load_gse106241():
    expr, meta = read_series_matrix(series_matrix("GSE106241"))
    e = probes_to_symbols(maybe_log2(expr, "GSE106241"), platform_table("GPL24170"),
                          symbol_col="GeneSymbol", name="GSE106241")
    sexcol = [c for c in meta.columns if c.startswith("sex")][0]
    m = pd.DataFrame({"donor": meta.index, "dx": "severity",
                      "age": pd.to_numeric(meta["age"], errors="coerce"),
                      "sex": meta[sexcol].map(sex_std),
                      "braak": pd.to_numeric(meta["braak stage"], errors="coerce"),
                      "region": "inferior temporal cortex"}, index=meta.index)
    return finish(e, m, "GSE106241", "GPL24170")


def load_gse1297():
    expr, meta = read_series_matrix(series_matrix("GSE1297"))
    e = probes_to_symbols(maybe_log2(expr, "GSE1297"), platform_table("GPL96"),
                          entrez_col="ENTREZ_GENE_ID", symbol_col="Gene Symbol", name="GSE1297")
    m = pd.DataFrame({"donor": meta.index, "dx": "severity",
                      "group": meta["group"],
                      "age": pd.to_numeric(meta["age"], errors="coerce"),
                      "sex": meta["sex"].map(sex_std),
                      "mmse": pd.to_numeric(meta["mmse"], errors="coerce"),
                      "nft": pd.to_numeric(meta["nft"], errors="coerce"),
                      "braak": pd.to_numeric(meta["braak"], errors="coerce"),
                      "region": "hippocampus"}, index=meta.index)
    return finish(e, m.drop(columns=["group"]), "GSE1297", "GPL96")


LOADERS = {"GSE132903": load_gse132903, "GSE15222": load_gse15222, "GSE29378": load_gse29378,
           "GSE122063": load_gse122063, "GSE36980": load_gse36980,
           "GSE106241": load_gse106241, "GSE1297": load_gse1297}
FAILED = {}
for name, fn in LOADERS.items():
    try:
        VAL[name] = fn()
    except Exception as ex:                       # one cohort failing must not stop the others
        import traceback; traceback.print_exc()
        FAILED[name] = f"{type(ex).__name__}: {ex}"
        print(f"!! {name} could not be loaded: {FAILED[name]}")
DX_COHORTS = [c for c in ["GSE132903", "GSE15222", "GSE29378", "GSE122063", "GSE36980"] if c in VAL]
SEV_COHORTS = [c for c in ["GSE106241", "GSE1297", "GSE29378"] if c in VAL]
pd.DataFrame([{"cohort": c, "donors": len(VAL[c][1]), "genes": VAL[c][0].shape[1],
               "AD": int((VAL[c][1].dx == "AD").sum()), "control": int((VAL[c][1].dx == "Control").sum()),
               "age_median": VAL[c][1].age.median(), "platform": VAL[c][1].platform.iloc[0]}
              for c in VAL]).to_csv(f"{OUT}/validation_cohorts.csv", index=False)
print("\nAD-vs-control cohorts:", DX_COHORTS, "| severity cohorts:", SEV_COHORTS, "| failed:", FAILED)


In [ ]:
# ======================================================================
# Per cohort: age- and sex-adjusted effect of AD on every gene (donor level)
# ======================================================================
def adjusted_de(E, meta):
    """OLS per gene on z-scored expression: expr ~ AD + age + sex.
    Returns coefficient of AD (in SD units), SE, t, p, df."""
    Z = (E - E.mean(0)) / E.std(0, ddof=0).replace(0, 1)
    cols = [np.ones(len(meta)), (meta.dx == "AD").astype(float).values]
    age = meta.age.fillna(meta.age.median()).values
    if np.nanstd(age) > 0:
        cols.append((age - age.mean()) / age.std())
    sx = (meta.sex == "male").astype(float).values
    if meta.sex.notna().any() and 0 < sx.mean() < 1:
        cols.append(sx)
    D = np.column_stack(cols)
    B, *_ = np.linalg.lstsq(D, Z.values, rcond=None)
    res = Z.values - D @ B
    df = len(D) - D.shape[1]
    s2 = (res ** 2).sum(0) / df
    XtXi = np.linalg.inv(D.T @ D)
    se = np.sqrt(s2 * XtXi[1, 1])
    t = B[1] / se
    p = 2 * stats.t.sf(np.abs(t), df)
    out = pd.DataFrame({"coef": B[1], "se": se, "t": t, "p": p}, index=E.columns)
    out["q"] = bh_fdr(np.nan_to_num(out.p.values, nan=1.0))
    out["df"] = df
    return out


DE = {}
for c in DX_COHORTS:
    E, meta = VAL[c]
    keep = meta.dx.isin(["AD", "Control"]).values
    DE[c] = adjusted_de(E[keep], meta[keep])
    DE[c].to_csv(f"{OUT}/de_{c}.csv")
    print(f"{c}: {keep.sum()} donors, {(DE[c].q < 0.05).sum():,} genes at FDR<0.05 (age/sex adjusted)")

# random-effects meta-analysis (DerSimonian-Laird) over the AD-vs-control cohorts
def meta_analyze(cohorts):
    genes_all = sorted(set.union(*[set(DE[c].index) for c in cohorts]))
    B = pd.DataFrame({c: DE[c].coef for c in cohorts}).reindex(genes_all)
    S = pd.DataFrame({c: DE[c].se for c in cohorts}).reindex(genes_all)
    W = 1 / S ** 2
    k = B.notna().sum(1)
    fe = (W * B).sum(1) / W.sum(1)
    Q = (W * (B.sub(fe, axis=0)) ** 2).sum(1)
    C_ = W.sum(1) - (W ** 2).sum(1) / W.sum(1)
    tau2 = ((Q - (k - 1)) / C_).clip(lower=0).fillna(0)
    Wr = 1 / (S ** 2 + tau2.values[:, None])
    pooled = (Wr * B).sum(1) / Wr.sum(1)
    pooled_se = np.sqrt(1 / Wr.sum(1))
    M = pd.DataFrame({"k": k, "pooled": pooled, "se": pooled_se,
                      "lo": pooled - 1.96 * pooled_se, "hi": pooled + 1.96 * pooled_se,
                      "z": pooled / pooled_se, "I2": ((Q - (k - 1)) / Q).clip(lower=0).fillna(0)})
    M["p"] = 2 * stats.norm.sf(np.abs(M.z))
    M.loc[M.k < 2, "p"] = np.nan
    M["q"] = np.nan
    ok = M.p.notna()
    M.loc[ok, "q"] = bh_fdr(M.loc[ok, "p"].values)
    return M


META = meta_analyze(list(DE))                                   # all AD-vs-control cohorts
NEW_COHORTS = [c for c in DE if c != "GSE132903"]
META_NEW = meta_analyze(NEW_COHORTS)                            # for the Core set: GSE132903 excluded
META.to_csv(f"{OUT}/meta_analysis_all_genes.csv")
META_NEW.to_csv(f"{OUT}/meta_analysis_new_cohorts_only.csv")
print(f"meta-analysis over {len(DE)} cohorts: {(META.q < 0.05).sum():,} genes at FDR<0.05")


In [ ]:
# ======================================================================
# The three sets against the matched-random and plain-DE baselines
# ======================================================================
directional = TRAIN_P[TRAIN_P < 0.10].index
bins = pd.qcut(TRAIN_LFC.loc[directional].abs().rank(method="first"), 10, labels=False)
BIN_OF = bins.to_dict()
BIN_MEMBERS = {b: bins.index[bins == b].values for b in range(10)}


def matched_null(gs, exclude, available):
    """One random set matched gene-for-gene on training effect-size decile, in the
    training direction of the random gene."""
    out = {}
    for g in gs:
        b = BIN_OF.get(g)
        pool = BIN_MEMBERS[b] if b is not None else np.array(list(directional))
        for _ in range(50):
            r = pool[RNG.randint(len(pool))]
            if r not in exclude and r in available and r not in out:
                out[r] = int(np.sign(TRAIN_LFC[r])); break
    return out


def plain_de(n, available):
    """The top-n genes by training differential expression (the simplest competitor)."""
    cand = TRAIN_P.loc[[g for g in directional if g in available]].sort_values().index[:n]
    return {g: int(np.sign(TRAIN_LFC[g])) for g in cand}


def score_set(gset, de):
    g = [x for x in gset if x in de.index]
    if not g:
        return dict(n=0, concordance=np.nan, replication=np.nan)
    s = np.array([gset[x] for x in g])
    same = np.sign(de.loc[g, "coef"].values) == s
    return dict(n=len(g), concordance=same.mean(), replication=(same & (de.loc[g, "q"].values < 0.05)).mean())


_ZCACHE = {}
def signature(gset, E):
    up = [g for g, s in gset.items() if s > 0 and g in E.columns]
    dn = [g for g, s in gset.items() if s < 0 and g in E.columns]
    key = (id(E), E.shape)
    if key not in _ZCACHE:
        _ZCACHE[key] = (E - E.mean(0)) / E.std(0, ddof=0).replace(0, 1)
    Z = _ZCACHE[key]
    return (Z[up].mean(1) if up else 0) - (Z[dn].mean(1) if dn else 0), len(up), len(dn)


_KEEP = {}
def sig_auc(gset, c):
    E, meta = VAL[c]
    keep = meta.dx.isin(["AD", "Control"]).values
    if c not in _KEEP:
        _KEEP[c] = E[keep]
    sc, nu, nd = signature(gset, _KEEP[c])
    y = (meta.dx[keep] == "AD").astype(int).values
    return roc_auc_score(y, sc), sc, y, meta[keep]


def lr_beyond_covariates(score, meta, y):
    """Likelihood-ratio test: AD ~ age + sex  vs  AD ~ age + sex + score."""
    cov = [meta.age.fillna(meta.age.median()).values, (meta.sex == "male").astype(float).values]
    X0 = np.column_stack(cov); X1 = np.column_stack(cov + [np.asarray(score)])
    def ll(X):
        clf = LogisticRegression(penalty=None, max_iter=5000).fit(X, y)
        p = np.clip(clf.predict_proba(X)[:, 1], 1e-12, 1 - 1e-12)
        return np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
    return stats.chi2.sf(2 * (ll(X1) - ll(X0)), 1)


rows = []
for setname, gset in SETS.items():
    cohorts = [c for c in DX_COHORTS if not (setname == "Core" and c == "GSE132903")]
    for c in cohorts + ["META"]:
        if c == "META":
            de = (META_NEW if setname == "Core" else META).rename(columns={"pooled": "coef"})
            avail = set(de.index[de.k >= 2])
        else:
            de = DE[c]; avail = set(de.index)
        obs = score_set(gset, de)
        r = {"set": setname, "cohort": c, **obs}
        # baselines
        null_c, null_r, null_auc = [], [], []
        excl = set(SETS["PLS-DA"]) | set(SETS["CVAE"])
        for _ in range(N_NULL if c == "META" else N_NULL // 2):
            ns = matched_null([g for g in gset if g in avail], excl, avail)
            sc = score_set(ns, de); null_c.append(sc["concordance"]); null_r.append(sc["replication"])
            if c != "META":
                null_auc.append(sig_auc(ns, c)[0])
        null_c, null_r = np.array(null_c), np.array(null_r)
        r.update(null_conc_mean=np.nanmean(null_c), null_conc_lo=np.nanpercentile(null_c, 2.5),
                 null_conc_hi=np.nanpercentile(null_c, 97.5),
                 p_conc=(1 + np.sum(null_c >= obs["concordance"])) / (1 + len(null_c)),
                 null_rep_mean=np.nanmean(null_r), null_rep_lo=np.nanpercentile(null_r, 2.5),
                 null_rep_hi=np.nanpercentile(null_r, 97.5),
                 p_rep=(1 + np.sum(null_r >= obs["replication"])) / (1 + len(null_r)))
        top = plain_de(obs["n"], avail)
        tsc = score_set(top, de)
        r.update(topDE_concordance=tsc["concordance"], topDE_replication=tsc["replication"])
        if c != "META":
            auc, sc, y, mk = sig_auc(gset, c)
            null_auc = np.array(null_auc)
            r.update(sig_auc=auc, null_auc_mean=null_auc.mean(), null_auc_lo=np.percentile(null_auc, 2.5),
                     null_auc_hi=np.percentile(null_auc, 97.5),
                     p_auc=(1 + np.sum(null_auc >= auc)) / (1 + len(null_auc)),
                     topDE_auc=sig_auc(top, c)[0],
                     lr_p_beyond_age_sex=lr_beyond_covariates(sc.values, mk, y))
        rows.append(r)
        print(f"{setname:7s} {c:10s} n={obs['n']:3d}  concord {obs['concordance']:.2f} (null {r['null_conc_mean']:.2f}, "
              f"p {r['p_conc']:.3g})  replic {obs['replication']:.2f} (null {r['null_rep_mean']:.2f}, p {r['p_rep']:.3g})"
              + (f"  AUC {r['sig_auc']:.3f} (null {r['null_auc_mean']:.3f}, p {r['p_auc']:.3g}; topDE {r['topDE_auc']:.3f})"
                 if c != "META" else ""))
SETRES = pd.DataFrame(rows)
SETRES.to_csv(f"{OUT}/set_validation.csv", index=False)

# per-gene table for the core and union sets
gene_rows = []
for g in sorted(set(SETS["PLS-DA"]) | set(SETS["CVAE"])):
    s = SETS["PLS-DA"].get(g, SETS["CVAE"].get(g))
    r = {"gene": g, "sets": "+".join(k for k in ("PLS-DA", "CVAE", "Core") if g in SETS[k]),
         "train_direction": "up" if s > 0 else "down", "train_lfc": TRAIN_LFC.get(g, np.nan)}
    for c in DX_COHORTS:
        if g in DE[c].index:
            r[f"{c}_coef"] = DE[c].loc[g, "coef"]; r[f"{c}_q"] = DE[c].loc[g, "q"]
    if g in META.index:
        for k_ in ("k", "pooled", "lo", "hi", "I2", "q"):
            r[f"meta_{k_}"] = META.loc[g, k_]
        r["meta_replicated"] = bool(np.sign(META.loc[g, "pooled"]) == s and META.loc[g, "q"] < 0.05)
    if g in META_NEW.index:
        for k_ in ("k", "pooled", "lo", "hi", "I2", "q"):
            r[f"metanew_{k_}"] = META_NEW.loc[g, k_]
    gene_rows.append(r)
GENES = pd.DataFrame(gene_rows)
GENES.to_csv(f"{OUT}/gene_validation_table.csv", index=False)
print(f"\n{GENES.meta_replicated.sum()} of {len(GENES)} selected genes replicate in the meta-analysis "
      f"(same direction, meta FDR<0.05)")


In [ ]:
# ======================================================================
# Dose-response: does the signature scale with neuropathology / cognition?
# ======================================================================
def partial_spearman(x, y, z):
    """Spearman correlation of x and y after removing a linear effect of z on the ranks."""
    ok = ~(np.isnan(x) | np.isnan(y) | np.isnan(z))
    rx, ry, rz = rankdata(x[ok]), rankdata(y[ok]), rankdata(z[ok])
    A = np.column_stack([np.ones(ok.sum()), rz])
    ex = rx - A @ np.linalg.lstsq(A, rx, rcond=None)[0]
    ey = ry - A @ np.linalg.lstsq(A, ry, rcond=None)[0]
    r, p = stats.pearsonr(ex, ey)
    return r, p, int(ok.sum())


dose_rows, dose_points = [], []
SEVERITY = [("GSE106241", "braak", +1), ("GSE1297", "braak", +1), ("GSE1297", "mmse", -1),
            ("GSE1297", "nft", +1), ("GSE29378", "braak", +1), ("GSE29378", "plaque", +1)]
for c, var, expected in SEVERITY:
    if c not in VAL or var not in VAL[c][1]:
        continue
    E, meta = VAL[c]
    sev = meta[var].values.astype(float)
    age = meta.age.fillna(meta.age.median()).values.astype(float)
    if np.isfinite(sev).sum() < 10:
        continue
    for setname, gset in SETS.items():
        sc, nu, nd = signature(gset, E)
        r, p, n = partial_spearman(sc.values, sev, age)
        excl = set(SETS["PLS-DA"]) | set(SETS["CVAE"])
        nulls = []
        for _ in range(N_NULL // 2):
            ns = matched_null([g for g in gset if g in E.columns], excl, set(E.columns))
            nulls.append(partial_spearman(signature(ns, E)[0].values, sev, age)[0] * expected)
        nulls = np.array(nulls)
        dose_rows.append({"cohort": c, "measure": var, "set": setname, "n": n, "n_up": nu, "n_down": nd,
                          "partial_rho": r, "p": p, "expected_sign": expected,
                          "null_mean": nulls.mean(), "null_lo": np.percentile(nulls, 2.5),
                          "null_hi": np.percentile(nulls, 97.5),
                          "p_vs_matched": (1 + np.sum(nulls >= r * expected)) / (1 + len(nulls))})
        for i in range(len(sc)):
            dose_points.append({"cohort": c, "measure": var, "set": setname, "score": sc.values[i],
                                "severity": sev[i], "age": age[i]})
        print(f"{c} {var:6s} {setname:7s} n={n:3d} partial rho={r:+.3f} (p {p:.2g}; "
              f"vs matched-random p {dose_rows[-1]['p_vs_matched']:.3g})")
DOSE = pd.DataFrame(dose_rows); DOSE.to_csv(f"{OUT}/dose_response.csv", index=False)
pd.DataFrame(dose_points).to_csv(f"{OUT}/dose_response_points.csv", index=False)


In [ ]:
# ======================================================================
# Report
# ======================================================================
L = ["INDEPENDENT VALIDATION OF THE FROZEN GENE LISTS", f"gene-list file SHA-256: "
     f"{hashlib.sha256(open(SEL_PATH, 'rb').read()).hexdigest()}", ""]
L.append("Cohorts: " + ", ".join(f"{c} ({len(VAL[c][1])} donors)" for c in VAL)
         + (f"; FAILED: {FAILED}" if FAILED else ""))
L += ["", "Set-level replication (adjusted for age and sex, donor level)."
      " Baselines: matched random = genes equally AD-associated in training; topDE = plain differential expression.", ""]
for _, r in SETRES.iterrows():
    line = (f"  {r.set:7s} {r.cohort:10s} n={r.n:3.0f}  same direction {r.concordance:5.1%} "
            f"[matched {r.null_conc_mean:5.1%}, p {r.p_conc:.3g}; topDE {r.topDE_concordance:5.1%}]  "
            f"replicated {r.replication:5.1%} [matched {r.null_rep_mean:5.1%}, p {r.p_rep:.3g}; topDE {r.topDE_replication:5.1%}]")
    if r.cohort != "META":
        line += (f"  signature AUC {r.sig_auc:.3f} [matched {r.null_auc_mean:.3f}, p {r.p_auc:.3g}; "
                 f"topDE {r.topDE_auc:.3f}; beyond age+sex p {r.lr_p_beyond_age_sex:.2g}]")
    L.append(line)
L += ["", "Dose-response (partial Spearman, age removed; sign expected: + for Braak/NFT/plaque, - for MMSE):"]
for _, r in DOSE.iterrows():
    L.append(f"  {r.cohort:10s} {r.measure:6s} {r.set:7s} n={r.n:3d} rho={r.partial_rho:+.3f} (p {r.p:.2g}); "
             f"matched-random p {r.p_vs_matched:.3g}")
core_g = GENES[GENES.sets.str.contains("Core")]
L += ["", f"Core genes ({len(core_g)}); their meta-analysis uses the new cohorts only (GSE132903 helped define them):"]
for _, r in core_g.iterrows():
    newc = [c for c in DX_COHORTS if c != "GSE132903" and f"{c}_coef" in r and pd.notna(r[f'{c}_coef'])]
    sgn = 1 if r.train_direction == "up" else -1
    same = sum(np.sign(r[f'{c}_coef']) == sgn for c in newc)
    sig = sum((np.sign(r[f'{c}_coef']) == sgn) and r[f'{c}_q'] < 0.05 for c in newc)
    L.append(f"  {r.gene:9s} {r.train_direction:4s}  same direction in {same}/{len(newc)} new cohorts, "
             f"significant in {sig}/{len(newc)}; meta (new cohorts) {r.get('metanew_pooled', np.nan):+.2f} "
             f"[{r.get('metanew_lo', np.nan):+.2f}, {r.get('metanew_hi', np.nan):+.2f}] "
             f"q={r.get('metanew_q', np.nan):.2g} I2={r.get('metanew_I2', np.nan):.0%}")
REPORT = "\n".join(L)
open(f"{OUT}/GENE_VALIDATION_REPORT.txt", "w").write(REPORT + "\n")
print(REPORT)


---

# Figures

The project's `figstyle` module, verbatim, then three figures drawn from the CSVs written above.


In [ ]:
# Point the shared style module at this notebook's output directory and
# ask it to render inline, then run it exactly as the paper build does.
import os
os.environ["PAPER_DATA"] = OUT
os.environ["PAPER_FIGS"] = f"{OUT}/figures"
os.environ["PAPER_INLINE"] = "1"

"""Shared visual language for every figure in the paper.

One module so that a panel letter, a tick, a bar gap or a significance bracket
looks the same wherever it appears -- the thing that separates a figure set from
a pile of plots. Everything here is layout and typography; the numbers come from
../data and are never touched.

Colour is the data-viz reference palette used unchanged: categorical slots 1-3
(blue / orange / aqua) for the three model arms, which is the documented
all-pairs colourblind-safe subset, and the documented blue-red diverging pair
with a neutral grey midpoint for up/down regulation. Identity never rests on
colour alone -- every series also carries a marker shape, a direct label or a
legend entry.
"""
import os
import warnings
from pathlib import Path

import matplotlib as mpl
# PAPER_INLINE is set when this same code runs inside the Kaggle notebook, where
# the figures must render in the output rather than only be written to disk.
INLINE = bool(os.environ.get("PAPER_INLINE"))
if not INLINE:
    mpl.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")

# Paths are overridable so that the notebook can point them at /kaggle/working
# and run this file unmodified -- one source of truth for every figure.
if "PAPER_DATA" in os.environ:
    DATA = Path(os.environ["PAPER_DATA"])
    FIG = Path(os.environ.get("PAPER_FIGS", DATA / "figures"))
else:
    ROOT = Path(__file__).resolve().parent.parent
    DATA = ROOT / "data"
    FIG = ROOT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

# Target printed width. The LaTeX text block is 6.27 in wide, so a figure drawn
# much wider than this is downscaled by \includegraphics and every font in it
# shrinks with the drawing -- the usual reason a figure set looks amateur on the
# page. Everything here is laid out at TW and lands at ~0.9 scale.
TW = 6.9

# ---- palette -------------------------------------------------------------
# A cool, low-chroma set in the register of a printed journal figure: navy,
# wine and violet rather than the primary blue / orange / green of a screen
# palette. The three model arms stay separable for the common forms of colour
# blindness -- they differ in hue *and* in lightness (navy 30%, wine 42%,
# violet 55%), so they also survive a greyscale print -- and identity never
# rests on colour alone, since every series carries a marker or a direct label.
PLS     = "#24486b"   # categorical slot 1 -- navy
CVAE_T  = "#8f3a4f"   # slot 2 -- wine
CVAE_N  = "#6f6396"   # slot 3 -- muted violet
AD      = "#8f3a4f"   # diverging warm pole -- wine
CTRL    = "#24486b"   # diverging cool pole -- navy
UP, DOWN = AD, CTRL
MID     = "#8a8a92"   # neutral midpoint ink, faintly cool
ACCENT  = "#2f7d8c"   # reserved for "found by both" -- petrol blue
INK     = "#17171c"
INK2    = "#54545e"
INK3    = "#8a8a92"
RULE    = "#c9c9d1"
GRID    = "#e9e9ef"
SURFACE = "#ffffff"

# Diverging ramp for probability surfaces: cool pole -> neutral grey -> warm
# pole. Deliberately light at both ends -- this ramp is only ever a background
# under data marks, and a saturated pole swallows them.
DIVERGING = LinearSegmentedColormap.from_list(
    "ctrl_ad", ["#9aabc0", "#bcc6d5", "#dadfe7", "#eeeef1",
                "#e8d3d9", "#d5aeb9", "#bf8b9a"])

# Sequential ramps for the confusion matrices, built from the two arm colours so
# the matrices sit inside the same muted palette as everything else instead of
# borrowing matplotlib's saturated "Blues"/"Oranges".
SEQ_PLS = LinearSegmentedColormap.from_list(
    "seq_pls", ["#ffffff", "#e7ecf2", "#c3d1e0", "#93aac6", "#5c7fa6", PLS, "#173252"])
SEQ_CVAE = LinearSegmentedColormap.from_list(
    "seq_cvae", ["#ffffff", "#f5e9ec", "#e6c9d1", "#d0a1ae", "#b06c81", CVAE_T, "#63242f"])

MODEL_COLOR = {"pls": PLS, "cvae_true": CVAE_T, "cvae_neutral": CVAE_N,
               "PLS-DA": PLS, "CVAE_true": CVAE_T, "CVAE_neutral": CVAE_N}
MODEL_LABEL = {"pls": "PLS-DA",
               "cvae_true": "CVAE (label-conditioned)",
               "cvae_neutral": "CVAE (label-agnostic)",
               "PLS-DA": "PLS-DA",
               "CVAE_true": "CVAE (label-conditioned)",
               "CVAE_neutral": "CVAE (label-agnostic)"}
GROUP_COLOR = {"AD": AD, "Control": CTRL}
GROUP_MARKER = {"AD": "^", "Control": "o"}

# database sources in the g:Profiler output
SRC_COLOR = {"GO:MF": "#24486b", "GO:BP": "#2f7d8c", "GO:CC": "#6f6396",
             "CORUM": "#8f3a4f", "KEGG": "#4b7ca8", "REAC": "#57516e",
             "WP": "#3f6d80", "MIRNA": "#a3798f", "TF": "#54545e", "HP": "#7a4a5e"}

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["STIXGeneral", "Latin Modern Roman", "DejaVu Serif", "Times New Roman"],
    "mathtext.fontset": "stix",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.titlelocation": "left",
    "axes.titlepad": 7,
    "axes.labelsize": 8,
    "axes.labelpad": 3.5,
    "axes.labelcolor": INK,
    "axes.edgecolor": RULE,
    "axes.linewidth": 0.6,
    "axes.grid": False,
    "grid.color": GRID,
    "grid.linewidth": 0.5,
    "axes.axisbelow": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.color": INK2, "ytick.color": INK2,
    "xtick.labelsize": 7, "ytick.labelsize": 7,
    "xtick.major.size": 2.6, "ytick.major.size": 2.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.pad": 2.5, "ytick.major.pad": 2.5,
    "legend.fontsize": 7,
    "legend.frameon": False,
    "legend.handlelength": 1.1,
    "legend.handletextpad": 0.5,
    "legend.columnspacing": 1.2,
    "legend.borderaxespad": 0.2,
    "lines.linewidth": 1.4,
    "lines.solid_capstyle": "round",
    "patch.linewidth": 0,
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "pdf.fonttype": 42,
})


# ---- primitives ----------------------------------------------------------
def save(fig, name):
    """Vector PDF for pdflatex, 400-dpi PNG for the pandoc/docx build."""
    for ext, kw in (("pdf", {}), ("png", {"dpi": 400})):
        fig.savefig(FIG / f"{name}.{ext}", bbox_inches="tight",
                    pad_inches=0.03, **kw)
    if INLINE:
        try:
            from IPython.display import display
            display(fig)
        except ImportError:      # running the notebook's cells outside a kernel
            pass
    plt.close(fig)
    print(f"  {FIG.name}/{name}.pdf + .png")


def panel(ax, letter, title=None, subtitle=None, dx=-0.13, dy=1.0):
    """Panel letter in sans-serif, hung outside the axes at a fixed offset so
    the letters line up down the figure regardless of tick-label width.

    Title and subtitle are both annotations rather than `set_title`, because a
    real title always sits at one fixed height and would land on top of the
    subtitle whenever both are present.
    """
    ax.annotate(letter, xy=(dx, dy), xycoords="axes fraction",
                xytext=(0, 2), textcoords="offset points",
                fontsize=9.5, fontweight="bold", family="DejaVu Sans",
                color=INK, ha="left", va="bottom", annotation_clip=False)
    if title:
        ax.annotate(title, xy=(0, 1.0), xycoords="axes fraction",
                    xytext=(0, 13 if subtitle else 3), textcoords="offset points",
                    fontsize=8.6, color=INK, ha="left", va="bottom",
                    annotation_clip=False)
    if subtitle:
        ax.annotate(subtitle, xy=(0, 1.0), xycoords="axes fraction",
                    xytext=(0, 3), textcoords="offset points",
                    fontsize=6.8, color=INK3, ha="left", va="bottom",
                    annotation_clip=False)


def ygrid(ax):
    ax.grid(axis="y", visible=True)
    ax.grid(axis="x", visible=False)


def xgrid(ax):
    ax.grid(axis="x", visible=True)
    ax.grid(axis="y", visible=False)


def despine(ax, left=True, bottom=True):
    ax.spines["left"].set_visible(left)
    ax.spines["bottom"].set_visible(bottom)


def load(name):
    return pd.read_csv(DATA / name)


def scal(df=None):
    """summary_scalars.csv -> dict, numeric where possible."""
    df = df if df is not None else load("summary_scalars.csv")
    out = {}
    for _, r in df.iterrows():
        try:
            out[r["key"]] = float(r["value"])
        except (TypeError, ValueError):
            out[r["key"]] = r["value"]
    return out


def stars(p):
    return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else "n.s."


def pmath(p, prefix="p"):
    """`$p = ...$` with a real times-ten-to-the exponent, not `1e-05`."""
    if p >= 1e-3:
        return f"${prefix} = {p:.3f}$"
    if not p > 1e-300:          # V7: with thousands of pairs a p-value can underflow to 0
        return f"${prefix} < 10^{{-300}}$"
    e = int(np.floor(np.log10(p)))
    return f"${prefix} = {p/10**e:.1f}\\times 10^{{{e}}}$"


def italic_ticks(ax, names, fontsize=6.0, axis="y"):
    """Italic gene symbols as real text, so hyphens stay hyphens and digits do
    not pick up math-italic spacing (`HLA-DQA2`, not `HLA - DQA2`)."""
    ticks = np.arange(len(names))
    if axis == "y":
        ax.set_yticks(ticks)
        ax.set_yticklabels(names, fontsize=fontsize, fontstyle="italic")
    else:
        ax.set_xticks(ticks)
        ax.set_xticklabels(names, fontsize=fontsize, fontstyle="italic")


def bracket(ax, x0, x1, y, text, tick=None, fontsize=7.2, color=INK):
    """Significance bracket that never overlaps the marks it spans."""
    tick = tick if tick is not None else abs(y) * 0.02 + 1e-9
    ax.plot([x0, x0, x1, x1], [y - tick, y, y, y - tick],
            color=color, lw=0.7, clip_on=False, solid_capstyle="butt")
    ax.text((x0 + x1) / 2, y + tick * 0.5, text, ha="center", va="bottom",
            fontsize=fontsize, color=color, clip_on=False)


def boxstrip(ax, values, pos, color, width=0.44, jitter=0.055, seed=0, size=4):
    """A box over its own points: the distribution and its summary together."""
    rs = np.random.RandomState(int(round(seed)))
    ax.scatter(rs.normal(pos, jitter, len(values)), values, s=size, color=color,
               alpha=0.28, linewidth=0, zorder=1, rasterized=True)
    bp = ax.boxplot([values], positions=[pos], widths=width, showfliers=False,
                    patch_artist=True, zorder=3, manage_ticks=False)
    bp["boxes"][0].set(facecolor=SURFACE, edgecolor=color, linewidth=1.2, alpha=0.95)
    for part in ("whiskers", "caps", "medians"):
        for a in bp[part]:
            a.set(color=color, linewidth=1.2)
    return bp


def swatches(labels_colors, marker="s", size=5):
    return [Line2D([], [], marker=marker, ls="", color=c, markersize=size, label=l)
            for l, c in labels_colors]


def repel_labels(ax, xs, ys, labels, fontsize=5.4, color=None, italic=True,
                 clearance=1.6, leader_color=None):
    """Place a label for every point without letting any two labels touch.

    A dependency-free replacement for adjustText: each label is tried at a ring
    of candidate offsets, in decreasing order of preference, and the first
    offset whose estimated box overlaps neither an already-placed label nor its
    own point is taken. Everything is measured in typographic points, which is
    exactly the unit `annotate` takes for `xytext`, so no round trip through
    data coordinates is needed and the result is stable at any figure size.
    """
    color = INK if color is None else color
    leader_color = RULE if leader_color is None else leader_color
    fig = ax.figure
    ppd = fig.dpi / 72.0                     # points -> display pixels
    pts = ax.transData.transform(np.column_stack([xs, ys]))

    # candidate offsets: near ring first, then progressively further out
    cands = []
    for radius in (7, 11, 16, 22, 30, 40, 52, 66, 82, 100):   # V7: wider search
        for ang in range(0, 360, 30):
            a = np.deg2rad(ang)
            cands.append((radius * np.cos(a), radius * np.sin(a)))

    placed = []                              # display-space boxes already taken
    leg = ax.get_legend()                    # a label under the key is unreadable
    if leg is not None:
        bb = leg.get_window_extent()
        placed.append((bb.x0, bb.y0, bb.x1, bb.y1))
    order = np.argsort(-np.asarray(ys))      # most significant genes get first pick

    def overlaps(box):
        x0, y0, x1, y1 = box
        for p0, q0, p1, q1 in placed:
            if x0 < p1 and p0 < x1 and y0 < q1 and q0 < y1:
                return True
        return False

    axbb = ax.get_window_extent()            # V7: labels must stay inside the axes
    inside = lambda b: (b[0] >= axbb.x0 and b[2] <= axbb.x1 and
                        b[1] >= axbb.y0 and b[3] <= axbb.y1)
    forced = 0
    for i in order:
        text = labels[i]
        w = len(text) * fontsize * 0.56 * ppd + clearance * ppd
        h = fontsize * 1.25 * ppd + clearance * ppd
        px, py = pts[i]
        boxes = []
        for dx, dy in cands:
            cx = px + dx * ppd + (w / 2 - clearance) * (1 if dx >= 0 else -1)
            cy = py + dy * ppd
            boxes.append(((cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2), dx, dy))
        ok = [t for t in boxes if inside(t[0]) and not overlaps(t[0])]
        if not ok:                           # never drop a gene: least-bad inside spot
            forced += 1
            ok = [t for t in boxes if inside(t[0])] or boxes
        for box, dx, dy in ok[:1]:
            if True:
                placed.append(box)
                ax.annotate(text, (xs[i], ys[i]), xytext=(dx, dy),
                            textcoords="offset points", fontsize=fontsize,
                            color=color, fontstyle="italic" if italic else "normal",
                            ha="left" if dx >= 0 else "right", va="center",
                            zorder=5, annotation_clip=False,
                            arrowprops=dict(arrowstyle="-", color=leader_color,
                                            lw=0.35, shrinkA=0.5, shrinkB=1.5))
    if forced:
        print(f"  repel_labels: {forced} label(s) had no collision-free spot")


def kde2d(xy, grid_x, grid_y, bw_scale=1.0):
    """Gaussian KDE evaluated on a grid, without a scipy import at call sites."""
    from scipy.stats import gaussian_kde
    k = gaussian_kde(xy.T)
    k.set_bandwidth(k.factor * bw_scale)
    XX, YY = np.meshgrid(grid_x, grid_y)
    return k(np.vstack([XX.ravel(), YY.ravel()])).reshape(XX.shape)

In [ ]:
# =====================================================================
# Figure G1 -- core genes in every independent cohort
# =====================================================================
def fig_core_genes():
    G = load("gene_validation_table.csv"); G = G[G.sets.str.contains("Core")].copy()
    if not len(G):
        print("no core genes"); return
    newc = [c for c in ["GSE15222", "GSE29378", "GSE122063", "GSE36980"] if f"{c}_coef" in G]
    sign = np.where(G.train_direction == "up", 1, -1)
    # meta over the NEW cohorts only, so GSE132903 (used to define the core) is excluded
    Bn = G[[f"{c}_coef" for c in newc]].values * sign[:, None]
    order = np.argsort(-np.nanmean(Bn, 1)); G = G.iloc[order]; Bn = Bn[order]; sign = sign[order]
    fig = plt.figure(figsize=(TW, 0.30 * len(G) + 1.9))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.25, 1.0], wspace=0.08)
    ax = fig.add_subplot(gs[0])
    lim = np.nanmax(np.abs(Bn)) if np.isfinite(Bn).any() else 1
    ax.imshow(Bn, cmap=LinearSegmentedColormap.from_list("c", [CTRL, SURFACE, ACCENT]),
              vmin=-lim, vmax=lim, aspect="auto")
    for i in range(len(G)):
        for j, c in enumerate(newc):
            q = G.iloc[i][f"{c}_q"]
            if pd.notna(q) and q < 0.05 and Bn[i, j] > 0:
                ax.text(j, i, "*", ha="center", va="center", fontsize=8, color=INK)
            elif np.isnan(Bn[i, j]):
                ax.text(j, i, "n/a", ha="center", va="center", fontsize=5, color=INK3)
    ax.set_xticks(range(len(newc))); ax.set_xticklabels(newc, rotation=30, ha="right", fontsize=6.6)
    italic_ticks(ax, [f"{g} ({d})" for g, d in zip(G.gene, G.train_direction)], 6.4)
    ax.tick_params(length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    panel(ax, "a", "Effect in the trained direction",
          "age/sex-adjusted, SD units; teal = replicates, * FDR $<0.05$", dx=-0.42)
    ax = fig.add_subplot(gs[1])
    y = np.arange(len(G))
    # summary over the new cohorts: mean and range of the trained-direction effect
    mu = np.nanmean(Bn, 1); lo = np.nanmin(Bn, 1); hi = np.nanmax(Bn, 1)
    ax.hlines(y, lo, hi, color=RULE, lw=1.2)
    ax.scatter(mu, y, s=26, color=ACCENT, zorder=3, edgecolor=SURFACE, linewidth=0.6)
    ax.axvline(0, color=INK3, lw=0.7, ls=(0, (3.5, 2.5)))
    ax.set_ylim(len(G) - 0.5, -0.5); ax.set_yticks([])
    ax.set_xlabel("effect in trained direction (SD)"); xgrid(ax)
    panel(ax, "b", "Mean and range over new cohorts", f"{len(newc)} cohorts, none used to pick these genes", dx=-0.05)
    save(fig, "figG1_core_genes")


fig_core_genes()


In [ ]:
# =====================================================================
# Figure G2 -- set-level replication against the baselines
# =====================================================================
def fig_sets():
    R = load("set_validation.csv")
    sets = [("PLS-DA", PLS, "o"), ("CVAE", CVAE_N, "s"), ("Core", ACCENT, "D")]
    cohorts = [c for c in R.cohort.unique() if c != "META"] + ["META"]
    fig = plt.figure(figsize=(TW, 7.4))
    gs = fig.add_gridspec(3, 1, hspace=0.75)
    for row, (obs, lo, hi, top, title) in enumerate((
            ("concordance", "null_conc_lo", "null_conc_hi", "topDE_concordance", "Same direction as in training"),
            ("replication", "null_rep_lo", "null_rep_hi", "topDE_replication", "Same direction and FDR $<0.05$"),
            ("sig_auc", "null_auc_lo", "null_auc_hi", "topDE_auc", "Signature score, ROC-AUC (no model)"))):
        ax = fig.add_subplot(gs[row])
        x = np.arange(len(cohorts))
        for j, (s, col, mk) in enumerate(sets):
            d = R[R.set == s].set_index("cohort").reindex(cohorts)
            if obs not in d or d[obs].isna().all():
                continue
            off = (j - 1) * 0.25
            ax.bar(x + off, d[hi] - d[lo], bottom=d[lo], width=0.21, color=GRID, edgecolor=RULE, lw=0.5)
            ax.scatter(x + off, d[top], marker="_", s=90, color=INK2, zorder=3, linewidth=1.4,
                       label="top-N plain DE" if j == 0 else None)
            ax.scatter(x + off, d[obs], marker=mk, s=28, color=col, zorder=4, edgecolor=SURFACE,
                       linewidth=0.6, label=s)
        if obs == "sig_auc":
            ax.axhline(0.5, color=INK3, lw=0.7, ls=(0, (3.5, 2.5)))
        ax.set_xticks(x); ax.set_xticklabels([c if c != "META" else "meta-analysis" for c in cohorts], fontsize=6.6)
        ax.set_xlim(-0.6, len(cohorts) - 0.4); ygrid(ax)
        ax.set_ylim(0.0 if obs != "sig_auc" else 0.3, 1.12)
        if row == 0:
            ax.legend(loc="lower left", ncol=4, fontsize=6.2, frameon=True, framealpha=0.92,
                      facecolor=SURFACE, edgecolor="none", columnspacing=1.0)
        panel(ax, "abc"[row], title, "grey bar = 95% range of matched-random sets; dash = top-N plain DE; "
                                     "Core is not tested on GSE132903 (its meta excludes it)", dx=-0.07)
    save(fig, "figG2_set_validation")


fig_sets()


In [ ]:
# =====================================================================
# Figure G3 -- dose-response with neuropathology and cognition
# =====================================================================
def fig_dose():
    D = load("dose_response.csv"); P = load("dose_response_points.csv")
    if not len(D):
        print("no severity data"); return
    panels = [(c, m) for c, m in D[["cohort", "measure"]].drop_duplicates().itertuples(index=False)]
    nc = 3; nr = int(np.ceil((len(panels) + 1) / nc))
    fig = plt.figure(figsize=(TW, 2.5 * nr + 0.6))
    gs = fig.add_gridspec(nr, nc, hspace=0.95, wspace=0.42)
    label = {"braak": "Braak stage", "mmse": "MMSE", "nft": "tangle density", "plaque": "plaque score"}
    for i, (c, m) in enumerate(panels):
        ax = fig.add_subplot(gs[i // nc, i % nc])
        p = P[(P.cohort == c) & (P.measure == m) & (P.set == "PLS-DA")]
        ax.scatter(p.severity + np.random.RandomState(i).uniform(-0.12, 0.12, len(p)) * (m != "nft"),
                   p.score, s=12, color=PLS, alpha=0.8, edgecolor=SURFACE, linewidth=0.4)
        if p.severity.nunique() > 1:
            z = np.polyfit(p.severity.dropna(), p.score[p.severity.notna()], 1)
            xx = np.linspace(p.severity.min(), p.severity.max(), 10)
            ax.plot(xx, np.polyval(z, xx), color=PLS, lw=1.0)
        ax.set_xlabel(label.get(m, m), fontsize=7); ax.set_ylabel("PLS-DA score" if i % nc == 0 else "", fontsize=7)
        ygrid(ax)
        d = D[(D.cohort == c) & (D.measure == m)].set_index("set")
        txt = "\n".join(f"{s}: $\\rho$={d.loc[s, 'partial_rho']:+.2f}" for s in ("PLS-DA", "CVAE", "Core") if s in d.index)
        ax.text(0.02, 0.98, txt, transform=ax.transAxes, va="top", fontsize=5.6, color=INK2)
        ax.set_ylim(ax.get_ylim()[0], ax.get_ylim()[1] + 0.45 * np.ptp(ax.get_ylim()))
        panel(ax, "abcdefgh"[i], c, f"n = {int(d.n.iloc[0])}; age-adjusted", dx=-0.3)
    ax = fig.add_subplot(gs[len(panels) // nc, len(panels) % nc])
    sets = [("PLS-DA", PLS, "o"), ("CVAE", CVAE_N, "s"), ("Core", ACCENT, "D")]
    yl = [f"{c}\n{label.get(m, m)}" for c, m in panels]
    for j, (s, col, mk) in enumerate(sets):
        for i, (c, m) in enumerate(panels):
            d = D[(D.cohort == c) & (D.measure == m) & (D.set == s)]
            if not len(d):
                continue
            d = d.iloc[0]; e = d.expected_sign
            yy = i + (j - 1) * 0.22
            ax.plot([d.null_lo, d.null_hi], [yy, yy], color=RULE, lw=2.4, solid_capstyle="butt")
            ax.scatter(d.partial_rho * e, yy, marker=mk, s=22, color=col, zorder=3,
                       label=s if i == 0 else None, edgecolor=SURFACE, linewidth=0.5)
    ax.axvline(0, color=INK3, lw=0.7, ls=(0, (3.5, 2.5)))
    ax.set_yticks(range(len(panels))); ax.set_yticklabels(yl, fontsize=5.4)
    ax.set_ylim(len(panels) - 0.5, -0.5); ax.set_xlabel("partial $\\rho$, expected direction", fontsize=7)
    ax.legend(loc="upper left", bbox_to_anchor=(1.03, 1.0), fontsize=6.0, frameon=False)
    panel(ax, "abcdefgh"[len(panels)], "All sets vs matched random", "grey = 95% range", dx=-0.55)
    save(fig, "figG3_dose_response")


fig_dose()


---

# Output manifest


In [ ]:
# every artefact this notebook wrote, as a single checkable manifest
import os

rows = []
for root, _, files in os.walk(OUT):
    for f in sorted(files):
        if f.startswith("__") or f.endswith(".ipynb"):
            continue
        p = os.path.join(root, f)
        rows.append({"file": os.path.relpath(p, OUT),
                     "bytes": os.path.getsize(p)})

manifest = pd.DataFrame(rows).sort_values("file").reset_index(drop=True)
manifest.to_csv(f"{OUT}/output_manifest.csv", index=False)
print(manifest.to_string(index=False))

empty = manifest[manifest.bytes == 0]
print(f"\n{len(manifest)} files, {manifest.bytes.sum()/1e6:.1f} MB")
if len(empty):
    print("EMPTY FILES -- something failed to render:")
    print(empty.to_string(index=False))
else:
    print("no empty files")